In [5]:
"""
GeoSync — DJI Flight Log → Sony EXIF GPS Injector
===================================================
Core innovations in this version:

1. AUTO-CALIBRATION of CAMERA_CLOCK_OFFSET
   Uses the drone's own isPhoto=1 events as GPS ground truth.
   Scans candidate offsets and finds the one that minimises
   the distance between each Sony photo's interpolated GPS
   and the nearest DJI trigger position.

2. SPEED-DISTANCE CORRELATION (fallback calibration)
   For each candidate offset: measures how well drone speed
   at photo-time correlates with distance to the next photo.
   Peaks sharply at the correct offset.

3. VELOCITY CORRECTION using xSpeed / ySpeed (world frame)
   Compensates for camera shutter latency using the actual
   East/North velocity vector — no heading-trig approximation.

4. POSITION UNCERTAINTY per photo in exported CSV.
"""

import os
import math
import shutil
import piexif
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ============================================================
# USER SETTINGS
# ============================================================

LOG_FILES = [
    r"/Users/snirtahasa/Thesis/DJI LOG/March-7-2021-12-56-05-Flight-Airdata.csv",
    r"/Users/snirtahasa/Thesis/DJI LOG/March-7-2021-13-15-59-Flight-Airdata.csv",
]
IMAGE_FOLDER  = r"/Users/snirtahasa/Thesis/2021-03-07/photos"
BACKUP_FOLDER = r"/Users/snirtahasa/Thesis/2021-03-07/photos_ORIGINAL_BACKUP"

# Starting guess (seconds). Auto-calibration will refine this.
CAMERA_CLOCK_OFFSET_SECONDS = -19.0

# Set only if camera is in local time (e.g. Israel UTC+2 → set 2).
# If camera is on UTC (common when synced to DJI GPS), leave 0.
CAMERA_TIMEZONE_OFFSET_HOURS = 0

# Camera mechanical shutter latency (ms). Drone keeps moving during this.
# Typical Sony mirrorless: 50–150 ms.
CAMERA_SHUTTER_LATENCY_MS = 100

# Auto-calibration search window around the starting guess (seconds).
CALIBRATION_SEARCH_WINDOW = 15.0   # scan ±15 s
CALIBRATION_STEP_COARSE   = 0.5    # coarse pass resolution
CALIBRATION_STEP_FINE     = 0.05   # fine pass resolution (around coarse best)

# Photos with positional uncertainty above this are flagged in the CSV.
UNCERTAINTY_FLAG_M = 2.0


# ============================================================
# UTILITIES
# ============================================================

def backup_images():
    if os.path.exists(BACKUP_FOLDER):
        print(f"  [backup] already exists — skipping.")
        return
    print(f"  [backup] copying to {BACKUP_FOLDER} ...")
    shutil.copytree(IMAGE_FOLDER, BACKUP_FOLDER)
    print(f"  [backup] done.")


def float_to_rational(value: float) -> tuple:
    """Decimal degrees → EXIF rational. 1e-6 denominator ≈ 1 cm precision."""
    if pd.isna(value):
        return ((0, 1), (0, 1), (0, 1))
    value   = abs(float(value))
    deg     = int(value)
    min_f   = (value - deg) * 60
    minutes = int(min_f)
    sec_f   = (min_f - minutes) * 60
    return ((deg, 1), (minutes, 1), (int(round(sec_f * 1_000_000)), 1_000_000))


def inject_gps_to_jpg(image_path: str, lat: float, lon: float, alt: float) -> bool:
    try:
        exif_dict = piexif.load(image_path)
    except Exception as e:
        print(f"    [ERROR] cannot read EXIF {os.path.basename(image_path)}: {e}")
        return False

    gps_ifd = {
        piexif.GPSIFD.GPSVersionID:    (2, 3, 0, 0),
        piexif.GPSIFD.GPSLatitudeRef:  b'N' if lat >= 0 else b'S',
        piexif.GPSIFD.GPSLatitude:     float_to_rational(abs(lat)),
        piexif.GPSIFD.GPSLongitudeRef: b'E' if lon >= 0 else b'W',
        piexif.GPSIFD.GPSLongitude:    float_to_rational(abs(lon)),
        piexif.GPSIFD.GPSAltitudeRef:  b'\x00' if alt >= 0 else b'\x01',
        piexif.GPSIFD.GPSAltitude:     (int(abs(alt) * 100), 100),
        piexif.GPSIFD.GPSMapDatum:     b'WGS-84',
    }
    try:
        exif_dict['GPS'] = gps_ifd
        exif_bytes = piexif.dump(exif_dict)
        piexif.remove(image_path)
        piexif.insert(exif_bytes, image_path)
        return True
    except Exception as e:
        print(f"    [ERROR] EXIF write failed {os.path.basename(image_path)}: {e}")
        return False


# ============================================================
# GPS INTERPOLATION HELPER
# ============================================================

def detect_flight_gaps(df_logs: pd.DataFrame,
                       gap_threshold_s: float = 30.0) -> list[tuple]:
    """
    Finds gaps between separate flights within the merged log.
    A gap is any interval between consecutive GPS points longer than
    gap_threshold_s seconds (typically happens between battery swaps).
    Returns a list of (gap_start, gap_end) as datetime64 pairs.
    """
    times  = df_logs['exact_time'].sort_values().values
    deltas = pd.Series(times[1:]) - pd.Series(times[:-1])
    gaps   = []
    for i, d in enumerate(deltas):
        if d > np.timedelta64(int(gap_threshold_s * 1e9), 'ns'):
            gaps.append((times[i], times[i + 1]))
    return gaps


def interpolate_gps(df_images_timed: pd.DataFrame,
                    df_logs: pd.DataFrame) -> pd.DataFrame:
    """
    Merge image timestamps into the GPS timeline and interpolate
    lat/lon/alt/xSpeed/ySpeed/speed for each image.

    Gap-aware: images that fall inside a between-flight gap
    (drone on ground between battery swaps) are excluded so they
    don't silently inherit the home-point coordinates.
    """
    # Detect gaps between separate flights (e.g. battery swap)
    gaps = detect_flight_gaps(df_logs, gap_threshold_s=30.0)

    cols_gps = ['exact_time', 'latitude', 'longitude',
                'altitude_above_seaLevel(meters)',
                'xSpeed(m/s)', 'ySpeed(m/s)', 'speed(m/s)']

    logs_s   = df_logs[cols_gps].copy();  logs_s['_type'] = 'GPS'
    images_s = df_images_timed[['exact_time', 'image_name']].copy()
    images_s['_type'] = 'IMAGE'

    combined = (
        pd.concat([logs_s, images_s])
          .sort_values('exact_time')
          .set_index('exact_time')
    )
    for col in ['latitude', 'longitude', 'altitude_above_seaLevel(meters)',
                'xSpeed(m/s)', 'ySpeed(m/s)', 'speed(m/s)']:
        combined[col] = combined[col].interpolate(method='time', limit_area='inside')

    # Null-out any image that falls inside a between-flight gap
    # so it is excluded by the subsequent dropna() call
    for gap_start, gap_end in gaps:
        mask = (
            (combined.index > gap_start) &
            (combined.index < gap_end) &
            (combined['_type'] == 'IMAGE')
        )
        combined.loc[mask, ['latitude', 'longitude']] = np.nan

    result = (
        combined[combined['_type'] == 'IMAGE']
          .dropna(subset=['latitude', 'longitude'])
          .reset_index()
    )
    return result, gaps


# ============================================================
# AUTO-CALIBRATION
# ============================================================

def _apply_offset(df_images_base: pd.DataFrame, offset_s: float) -> pd.DataFrame:
    df = df_images_base.copy()
    df['exact_time'] = (
        df['base_time']
        + pd.to_timedelta(df['subsec'], unit='s')
        + pd.to_timedelta(offset_s, unit='s')
    ).astype('datetime64[ns]')
    return df


def score_isphoto(offset_s: float,
                  df_images_base: pd.DataFrame,
                  df_logs: pd.DataFrame,
                  isphoto_rows: pd.DataFrame) -> float:
    """
    For each DJI isPhoto=1 event we know:
      - exact UTC time the drone trigger fired
      - exact GPS position at that moment

    For a given clock offset, find the Sony photo closest in time
    to each trigger event and measure how far its interpolated GPS
    is from the trigger GPS.

    Returns mean error in meters (lower = better offset).
    Only valid when len(isphoto_rows) >= 2.
    """
    if len(isphoto_rows) < 2:
        return np.inf

    df_img = _apply_offset(df_images_base, offset_s)
    result, _ = interpolate_gps(df_img, df_logs)
    if len(result) < 3:
        return np.inf

    # Build a lookup: image_name → interpolated GPS
    gps_lookup = result.set_index('image_name')[['latitude', 'longitude']].to_dict('index')

    # For each isPhoto event, find the nearest Sony photo (by time)
    img_times = df_img.set_index('exact_time')['image_name']
    errors = []
    for _, ev in isphoto_rows.iterrows():
        # Find closest Sony image in time to this DJI trigger
        t_dji = ev['exact_time']
        idx   = (img_times.index - t_dji).abs().argmin()
        img   = img_times.iloc[idx]
        if img not in gps_lookup:
            continue
        g     = gps_lookup[img]
        # Haversine-lite error (metres)
        dlat  = (g['latitude']  - ev['latitude'])  * 111_111
        dlon  = (g['longitude'] - ev['longitude']) * 111_111 * math.cos(math.radians(ev['latitude']))
        errors.append(math.sqrt(dlat**2 + dlon**2))

    return float(np.mean(errors)) if errors else np.inf


def score_speed_distance(offset_s: float,
                         df_images_base: pd.DataFrame,
                         df_logs: pd.DataFrame) -> float:
    """
    Fallback calibration: measures correlation between drone speed at
    photo-capture time and the GPS distance to the NEXT photo.

    When the offset is correct:
      fast drone  → large inter-photo gap
      slow drone  → small inter-photo gap
    → Pearson correlation is maximised.

    Returns negative correlation (so minimising == finding best offset).
    """
    df_img = _apply_offset(df_images_base, offset_s)
    result, _ = interpolate_gps(df_img, df_logs)
    if len(result) < 5:
        return 0.0

    lats   = result['latitude'].values
    lons   = result['longitude'].values
    speeds = result['speed(m/s)'].values

    d_lat  = np.diff(lats) * 111_111
    d_lon  = np.diff(lons) * 111_111 * np.cos(np.radians(np.mean(lats)))
    dists  = np.sqrt(d_lat**2 + d_lon**2)
    avg_sp = (speeds[:-1] + speeds[1:]) / 2

    if np.std(dists) < 1e-6 or np.std(avg_sp) < 1e-6:
        return 0.0
    corr = np.corrcoef(avg_sp, dists)[0, 1]
    return -float(corr) if not np.isnan(corr) else 0.0


def auto_calibrate(df_images_base: pd.DataFrame,
                   df_logs: pd.DataFrame,
                   isphoto_rows: pd.DataFrame,
                   manual_offset: float) -> float:
    """
    Two-pass offset search:
      Pass 1 (coarse): scan ±CALIBRATION_SEARCH_WINDOW in 0.5 s steps
      Pass 2 (fine):   scan ±2 s around coarse best in 0.05 s steps

    Uses isPhoto score when available (≥2 triggers), otherwise
    falls back to speed-distance correlation.
    """
    use_isphoto = len(isphoto_rows) >= 2
    method_name = "isPhoto anchor" if use_isphoto else "speed-distance correlation"
    print(f"  calibration method : {method_name}")

    score_fn = (
        lambda o: score_isphoto(o, df_images_base, df_logs, isphoto_rows)
        if use_isphoto
        else lambda o: score_speed_distance(o, df_images_base, df_logs)
    )

    # ── coarse pass ──────────────────────────────────────
    coarse_offsets = np.arange(
        manual_offset - CALIBRATION_SEARCH_WINDOW,
        manual_offset + CALIBRATION_SEARCH_WINDOW + CALIBRATION_STEP_COARSE,
        CALIBRATION_STEP_COARSE
    )
    coarse_scores = [score_fn(o) for o in coarse_offsets]
    best_coarse   = coarse_offsets[int(np.argmin(coarse_scores))]

    # ── fine pass ────────────────────────────────────────
    fine_offsets = np.arange(
        best_coarse - 2.0,
        best_coarse + 2.0 + CALIBRATION_STEP_FINE,
        CALIBRATION_STEP_FINE
    )
    fine_scores = [score_fn(o) for o in fine_offsets]
    best_fine   = fine_offsets[int(np.argmin(fine_scores))]

    delta = best_fine - manual_offset
    print(f"  manual offset      : {manual_offset:.1f} s")
    print(f"  auto-calibrated    : {best_fine:.2f} s  (Δ = {delta:+.2f} s)")
    if use_isphoto:
        print(f"  best anchor error  : {min(fine_scores):.1f} m  "
              f"(across {len(isphoto_rows)} DJI trigger events)")
    return best_fine


# ============================================================
# MAIN
# ============================================================

def run_smart_geosync():
    print("=" * 60)
    print("  GeoSync  ·  Auto-Calibrated  ·  Velocity-Corrected")
    print("=" * 60)

    backup_images()

    # ── 1. Load GPS logs ──────────────────────────────────────────
    print("\n[1/5] Loading DJI flight logs...")
    df_list = []
    for f in LOG_FILES:
        if os.path.exists(f):
            df = pd.read_csv(f)
            df.columns = df.columns.str.strip()   # remove DJI leading spaces
            df_list.append(df)
            print(f"  ✔ {os.path.basename(f)}")
        else:
            print(f"  ✗ not found: {f}")

    if not df_list:
        print("[ERROR] No logs found. Exiting."); return

    df_logs = pd.concat(df_list, ignore_index=True)
    df_logs['exact_time'] = (
        pd.to_datetime(df_logs['datetime(utc)'])
        + pd.to_timedelta(df_logs['time(millisecond)'], unit='ms')
    )
    df_logs = (
        df_logs
          .sort_values('exact_time')
          .dropna(subset=['latitude', 'longitude'])
          .astype({'exact_time': 'datetime64[ns]'})
    )
    gps_start = df_logs['exact_time'].iloc[0]
    gps_end   = df_logs['exact_time'].iloc[-1]
    print(f"  GPS window  : {gps_start}  →  {gps_end} UTC")
    print(f"  GPS points  : {len(df_logs):,}  (10 Hz)")

    # Extract DJI trigger events (isPhoto=1) — our calibration anchors
    isphoto_rows = df_logs[df_logs['isPhoto'] == 1][
        ['exact_time', 'latitude', 'longitude', 'speed(m/s)']
    ].reset_index(drop=True)
    print(f"  isPhoto events: {len(isphoto_rows)} "
          f"(GPS ground-truth anchors for calibration)")
    for _, r in isphoto_rows.iterrows():
        print(f"    {r['exact_time']}  lat={r['latitude']:.6f}  "
              f"lon={r['longitude']:.6f}  speed={r['speed(m/s)']:.2f} m/s")

    # ── 2. Read image EXIF timestamps ─────────────────────────────
    print("\n[2/5] Reading image EXIF timestamps...")
    image_files = sorted([
        f for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith('.jpg')
    ])
    print(f"  Found {len(image_files)} JPG files")

    images_data, errors = [], []
    for img in image_files:
        path = os.path.join(IMAGE_FOLDER, img)
        try:
            exif_dict = piexif.load(path)
            exif_ifd  = exif_dict.get("Exif", {})
            dt_bytes  = exif_ifd.get(piexif.ExifIFD.DateTimeOriginal)
            if not dt_bytes:
                errors.append(f"missing DateTimeOriginal: {img}"); continue
            dt = datetime.strptime(dt_bytes.decode().strip('\x00'), "%Y:%m:%d %H:%M:%S")

            # SubSecTimeOriginal: Sony Alpha writes real sub-second precision
            subsec = 0.0
            ss     = exif_ifd.get(piexif.ExifIFD.SubSecTimeOriginal)
            if ss:
                s = ss.decode().strip('\x00')
                if s.isdigit():
                    subsec = float("0." + s)

            images_data.append({'image_name': img, 'base_time': dt, 'subsec': subsec})
        except Exception as e:
            errors.append(f"EXIF error {img}: {e}")

    if errors:
        print(f"  ⚠  {len(errors)} images with issues (first 3):")
        for e in errors[:3]: print(f"     {e}")
    if not images_data:
        print("[ERROR] No valid images found. Exiting."); return

    df_images = pd.DataFrame(images_data).sort_values('base_time').reset_index(drop=True)

    # Sub-second timing
    has_subsec = (df_images['subsec'] > 0).any()
    if has_subsec:
        print(f"  ✅ SubSecTimeOriginal present — sub-second timestamps used")
    else:
        print(f"  ℹ  No SubSecTimeOriginal — distributing evenly within each second")
        df_images['count'] = df_images.groupby('base_time').cumcount()
        df_images['total'] = df_images.groupby('base_time')['image_name'].transform('count')
        df_images['subsec'] = df_images['count'] / df_images['total']

    # Apply timezone offset to base_time (not the clock drift)
    tz_delta = CAMERA_TIMEZONE_OFFSET_HOURS * 3600
    df_images['base_time'] = df_images['base_time'] + pd.to_timedelta(tz_delta, unit='s')

    # ── 3. Auto-calibrate clock offset ───────────────────────────
    print("\n[3/5] Auto-calibrating CAMERA_CLOCK_OFFSET...")
    best_offset = auto_calibrate(
        df_images, df_logs, isphoto_rows, CAMERA_CLOCK_OFFSET_SECONDS
    )

    # Apply the calibrated offset
    df_images['exact_time'] = (
        df_images['base_time']
        + pd.to_timedelta(df_images['subsec'], unit='s')
        + pd.to_timedelta(best_offset, unit='s')
    ).astype('datetime64[ns]')

    # Warn about out-of-range images
    oor = df_images[
        (df_images['exact_time'] < gps_start) | (df_images['exact_time'] > gps_end)
    ]
    if len(oor):
        print(f"\n  ⚠  {len(oor)} images outside GPS window — will be skipped")

    # ── 4. Interpolate + velocity correction ─────────────────────
    print("\n[4/5] Interpolating GPS + applying velocity correction...")
    final_images, gaps = interpolate_gps(df_images, df_logs)

    if gaps:
        print(f"  Detected {len(gaps)} between-flight gap(s):")
        for gs, ge in gaps:
            dur = (pd.Timestamp(ge) - pd.Timestamp(gs)).total_seconds()
            print(f"    {pd.Timestamp(gs).strftime('%H:%M:%S')} → "
                  f"{pd.Timestamp(ge).strftime('%H:%M:%S')} UTC  "
                  f"({dur:.0f} s on ground — images in gap excluded)")

    skipped = len(df_images) - len(final_images)
    print(f"  {len(final_images)} images matched,  {skipped} skipped")

    # Velocity correction: compensate for camera shutter latency
    # Position at shutter-open = GPS position + velocity × latency
    # Uses world-frame xSpeed (East) and ySpeed (North) directly.
    dt_s = CAMERA_SHUTTER_LATENCY_MS / 1000.0

    def apply_correction(row):
        v_east  = row['xSpeed(m/s)']
        v_north = row['ySpeed(m/s)']
        lat     = row['latitude']
        dlat = (v_north * dt_s) / 111_111
        dlon = (v_east  * dt_s) / (111_111 * math.cos(math.radians(lat)))
        spd  = row['speed(m/s)']
        # Uncertainty = ½s residual clock drift × speed (conservative estimate)
        uncertainty = round(spd * 0.5, 2)
        return pd.Series({
            'lat_final':  lat + dlat,
            'lon_final':  row['longitude'] + dlon,
            'speed_mps':  round(spd, 2),
            'uncertainty_m': uncertainty,
            'flag': 'HIGH' if uncertainty > UNCERTAINTY_FLAG_M else 'OK',
        })

    corrections = final_images.apply(apply_correction, axis=1)
    final_images = pd.concat([final_images, corrections], axis=1)

    n_high = (final_images['flag'] == 'HIGH').sum()
    if n_high:
        print(f"  ⚠  {n_high} photos flagged HIGH uncertainty "
              f"(speed > {UNCERTAINTY_FLAG_M / 0.5:.0f} m/s at capture)")

    # ── 5. Inject EXIF ───────────────────────────────────────────
    print(f"\n[5/5] Injecting GPS EXIF into {len(final_images)} images...")
    success, failed = 0, 0
    for _, row in final_images.iterrows():
        path = os.path.join(IMAGE_FOLDER, row['image_name'])
        ok   = inject_gps_to_jpg(path,
                                  row['lat_final'],
                                  row['lon_final'],
                                  row['altitude_above_seaLevel(meters)'])
        if ok: success += 1
        else:  failed  += 1

    # ── Export CSV ────────────────────────────────────────────────
    csv_path = os.path.join(IMAGE_FOLDER, "gps_coordinates.csv")
    final_images[[
        'image_name', 'exact_time',
        'lat_final', 'lon_final',
        'altitude_above_seaLevel(meters)',
        'speed_mps', 'uncertainty_m', 'flag'
    ]].rename(columns={
        'lat_final': 'latitude',
        'lon_final': 'longitude',
        'altitude_above_seaLevel(meters)': 'altitude_m',
        'exact_time': 'datetime_utc',
    }).to_csv(csv_path, index=False)

    # ── Summary ───────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print(f"  ✅ Success          : {success} images")
    if failed:   print(f"  ❌ Failed           : {failed} images")
    if skipped:  print(f"  ⏭  Skipped          : {skipped} (outside GPS window)")
    if n_high:   print(f"  ⚠  High uncertainty : {n_high} (fast flight, see CSV flag)")
    print(f"\n  Auto-calibrated offset: {best_offset:.2f} s")
    print(f"  Velocity correction  : {CAMERA_SHUTTER_LATENCY_MS} ms latency")
    print(f"  CSV: {csv_path}")
    print("=" * 60)
    print("\n  ArcGIS: GeoTagged Photos to Points | GCS_WGS_1984")
    print("  Tip: colour-code points by 'flag' column to see uncertainty")


if __name__ == "__main__":
    run_smart_geosync()

  GeoSync  ·  Auto-Calibrated  ·  Velocity-Corrected
  [backup] already exists — skipping.

[1/5] Loading DJI flight logs...
  ✔ March-7-2021-12-56-05-Flight-Airdata.csv
  ✔ March-7-2021-13-15-59-Flight-Airdata.csv
  GPS window  : 2021-03-07 10:56:05.100000  →  2021-03-07 11:34:24.600000 UTC
  GPS points  : 15,986  (10 Hz)
  isPhoto events: 0 (GPS ground-truth anchors for calibration)

[2/5] Reading image EXIF timestamps...
  Found 960 JPG files
  ℹ  No SubSecTimeOriginal — distributing evenly within each second

[3/5] Auto-calibrating CAMERA_CLOCK_OFFSET...
  calibration method : speed-distance correlation


TypeError: '<' not supported between instances of 'function' and 'function'

In [ ]:
 from osgeo import gdal

# עדכן את הנתיב המדויק לקובץ החיזוי
file_path = r"/Users/snirtahasa/Thesis/2021-03-07/Preds/rasters/DSC07999.tif"

ds = gdal.Open(file_path)
if ds:
    geo_transform = ds.GetGeoTransform()
    projection = ds.GetProjection()
    
    print("--- Metadata Check ---")
    if geo_transform == (0.0, 1.0, 0.0, 0.0, 0.0, 1.0) and not projection:
        print("❌ אין נתוני מיקום. התמונה בנקודת 0,0 חסרת קואורדינטות.")
    else:
        print(f"✅ יש נתוני מיקום!")
        print(f"GeoTransform: {geo_transform}")
        print(f"Projection: {projection[:50]}...")
else:
    print("Failed to open file.")

--- Metadata Check ---
✅ יש נתוני מיקום!
GeoTransform: (669847.1024566873, 0.02, 0.0, 3509826.2340989676, 0.0, -0.02)
Projection: PROJCS["WGS 84 / UTM zone 36N",GEOGCS["WGS 84",DAT...


In [ ]:
/Users/snirtahasa/Thesis/2021-03-07/DSC07999.tif

In [1]:
import piexif
import os

# בחר תמונה אחת ספציפית מהתיקייה שלך
IMAGE_PATH = r"/Users/snirtahasa/Thesis/2021-03-07/DSC07999.JPG" 

def dump_all_exif():
    print(f"Analyzing {os.path.basename(IMAGE_PATH)}...\n")
    try:
        exif_dict = piexif.load(IMAGE_PATH)
        
        # מעבר על כל קטגוריות המידע ב-EXIF
        for ifd_name in ["0th", "Exif", "GPS", "1st"]:
            if not exif_dict[ifd_name]:
                continue
                
            print(f"--- {ifd_name} Data ---")
            for tag_id, value in exif_dict[ifd_name].items():
                # מנסה למצוא את השם המילולי של התגית
                tag_name = piexif.TAGS[ifd_name].get(tag_id, {}).get("name", str(tag_id))
                
                # ניקוי פורמטים כדי שיהיה קריא לעין
                if isinstance(value, bytes):
                    try:
                        value = value.decode('utf-8').strip('\x00')
                    except:
                        value = "<Binary Data>"
                
                print(f"{tag_name}: {value}")
            print("\n")
            
    except Exception as e:
        print(f"Error reading EXIF: {e}")

if __name__ == "__main__":
    dump_all_exif()

Analyzing DSC07999.JPG...

--- 0th Data ---
ImageDescription:                                
Make: SONY
Model: ILCE-6000
Orientation: 3
XResolution: (350, 1)
YResolution: (350, 1)
ResolutionUnit: 2
Software: ILCE-6000 v3.21
DateTime: 2021:03:07 11:04:23
YCbCrPositioning: 2
ExifTag: 364
PrintImageMatching: <Binary Data>


--- Exif Data ---
ExposureTime: (1, 1250)
FNumber: (40, 10)
ExposureProgram: 4
ISOSpeedRatings: 100
SensitivityType: 2
RecommendedExposureIndex: 100
ExifVersion: 0230
DateTimeOriginal: 2021:03:07 11:04:23
DateTimeDigitized: 2021:03:07 11:04:23
ComponentsConfiguration: 
CompressedBitsPerPixel: (2, 1)
BrightnessValue: (24604, 2560)
ExposureBiasValue: (0, 10)
MaxApertureValue: (760, 256)
MeteringMode: 5
LightSource: 1
Flash: 16
FocalLength: (160, 10)
MakerNote: <Binary Data>
UserComment: 
FlashpixVersion: 0100
ColorSpace: 1
PixelXDimension: 6000
PixelYDimension: 4000
InteroperabilityTag: 38154
FileSource: 
SceneType: 
CustomRendered: 0
ExposureMode: 0
WhiteBalance: 

In [2]:
import piexif
import os

IMAGE_PATH = r"/Users/snirtahasa/Thesis/2021-03-07/DSC07999.JPG" 
OUTPUT_FILE = r"/Users/snirtahasa/Thesis/2021-03-07/exif_dump.txt"

def save_all_exif():
    print("שומר את כל הנתונים לקובץ טקסט...")
    try:
        exif_dict = piexif.load(IMAGE_PATH)
        
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            f.write(f"Full EXIF Dump for {os.path.basename(IMAGE_PATH)}\n")
            f.write("="*50 + "\n\n")
            
            for ifd_name in ["0th", "Exif", "GPS", "1st"]:
                if not exif_dict[ifd_name]:
                    continue
                    
                f.write(f"--- {ifd_name} Data ---\n")
                for tag_id, value in exif_dict[ifd_name].items():
                    tag_name = piexif.TAGS[ifd_name].get(tag_id, {}).get("name", str(tag_id))
                    
                    # טיפול בנתונים ארוכים במיוחד או בינאריים
                    if isinstance(value, bytes):
                        if len(value) > 150:
                            val_str = f"<Binary Data - Length: {len(value)} bytes>"
                        else:
                            try:
                                val_str = value.decode('utf-8', errors='ignore').strip('\x00')
                            except:
                                val_str = f"<Binary Data: {value}>"
                    else:
                        val_str = str(value)
                    
                    f.write(f"{tag_name}: {val_str}\n")
                f.write("\n")
                
        print(f"הקובץ נשמר בהצלחה! פתח את הקובץ הבא כדי לראות את הכל:")
        print(OUTPUT_FILE)
            
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    save_all_exif()

שומר את כל הנתונים לקובץ טקסט...
הקובץ נשמר בהצלחה! פתח את הקובץ הבא כדי לראות את הכל:
/Users/snirtahasa/Thesis/2021-03-07/exif_dump.txt


In [3]:
import os
import piexif
import pandas as pd
from datetime import datetime

IMAGE_FOLDER = r"/Users/snirtahasa/Thesis/2021-03-07"

def analyze_all_timings():
    image_files = [f for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith('.jpg')]
    image_files.sort()
    
    data = []
    print(f"סורק {len(image_files)} תמונות... אנא המתן.\n")
    
    # שלב 1: איסוף הזמנים
    for img in image_files:
        path = os.path.join(IMAGE_FOLDER, img)
        try:
            exif_dict = piexif.load(path)
            # משיכת תאריך ושעת הצילום המקוריים
            dt_str = exif_dict["Exif"].get(piexif.ExifIFD.DateTimeOriginal)
            
            if dt_str:
                # ניקוי הפורמט הבינארי או תווים מיותרים
                if isinstance(dt_str, bytes):
                    dt_str = dt_str.decode('utf-8').strip('\x00')
                
                # המרה של הפורמט לאובייקט datetime של פייתון
                dt = datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S")
                data.append({'image_name': img, 'time': dt})
        except Exception as e:
            # במקרה שתמונה מסוימת פגומה או חסרת EXIF
            print(f"שגיאה בקריאת התמונה {img}: {e}")
            
    if not data:
        print("לא נמצאו נתוני זמן (EXIF) באף תמונה.")
        return
        
    # שלב 2: בניית דאטה-פריים (טבלה חכמה)
    df = pd.DataFrame(data)
    df = df.sort_values('time').reset_index(drop=True)
    
    # חישוב הפער בשניות בין כל תמונה לתמונה שבאה אחריה
    df['time_diff_seconds'] = df['time'].diff().dt.total_seconds()
    
    # שלב 3: הדפסת הדוח
    print("="*50)
    print("📊 דוח ניתוח זמני צילום מה-EXIF")
    print("="*50)
    print(f"סה\"כ תמונות תקינות שנותחו: {len(df)}")
    print(f"זמן תמונה ראשונה:          {df['time'].iloc[0]}")
    print(f"זמן תמונה אחרונה:          {df['time'].iloc[-1]}")
    
    total_duration = df['time'].iloc[-1] - df['time'].iloc[0]
    print(f"משך צילום כולל:            {total_duration}")
    
    # חישוב ממוצע מתעלם מפערים ענקיים (כמו הפסקות להחלפת סוללה) כדי לתת את קצב הצילום האמיתי
    normal_diffs = df[df['time_diff_seconds'] < 10]
    if not normal_diffs.empty:
        avg_diff = normal_diffs['time_diff_seconds'].mean()
        print(f"מרווח זמן ממוצע (נקי):     {avg_diff:.2f} שניות בין תמונות")
    print("\n")
    
    print("="*50)
    print("⚠️ איתור פערים ואנומליות (> 3 שניות)")
    print("="*50)
    
    # זיהוי פערים גדולים מ-3 שניות
    gaps = df[df['time_diff_seconds'] > 3.0]
    
    if gaps.empty:
        print("✅ הצילום היה מושלם! רצף עקבי ללא קפיצות זמן חריגות.")
    else:
        print(f"נמצאו {len(gaps)} קפיצות זמן חריגות:")
        for idx, row in gaps.iterrows():
            prev_img = df.iloc[idx-1]['image_name']
            curr_img = row['image_name']
            gap_sec = row['time_diff_seconds']
            prev_time = df.iloc[idx-1]['time']
            curr_time = row['time']
            
            print(f"-> קפיצה של {gap_sec} שניות!")
            print(f"   מהתמונה: {prev_img} בשעה {prev_time.strftime('%H:%M:%S')}")
            print(f"   לתמונה:  {curr_img} בשעה {curr_time.strftime('%H:%M:%S')}\n")

if __name__ == "__main__":
    analyze_all_timings()

סורק 960 תמונות... אנא המתן.

📊 דוח ניתוח זמני צילום מה-EXIF
סה"כ תמונות תקינות שנותחו: 960
זמן תמונה ראשונה:          2021-03-07 10:56:19
זמן תמונה אחרונה:          2021-03-07 11:27:39
משך צילום כולל:            0 days 00:31:20
מרווח זמן ממוצע (נקי):     1.96 שניות בין תמונות


⚠️ איתור פערים ואנומליות (> 3 שניות)
✅ הצילום היה מושלם! רצף עקבי ללא קפיצות זמן חריגות.


In [15]:
import piexif
import os

IMAGE_PATH = r"/Users/snirtahasa/Thesis/2021-03-07/DSC07845.JPG" 

def check_gps_exif():
    print(f"בודק נתוני GPS עבור: {os.path.basename(IMAGE_PATH)}\n")
    try:
        exif_dict = piexif.load(IMAGE_PATH)
        gps_data = exif_dict.get("GPS", {})
        
        if not gps_data:
            print("❌ לא נמצאו נתוני GPS בתמונה הזו!")
            return
            
        print("✅ נתוני GPS נמצאו! הנה הפירוט:")
        for tag_id, value in gps_data.items():
            tag_name = piexif.TAGS["GPS"].get(tag_id, {}).get("name", str(tag_id))
            print(f"{tag_name}: {value}")
            
    except Exception as e:
        print(f"שגיאה בקריאת הקובץ: {e}")

if __name__ == "__main__":
    check_gps_exif()

בודק נתוני GPS עבור: DSC07845.JPG

✅ נתוני GPS נמצאו! הנה הפירוט:
GPSVersionID: (2, 2, 0, 0)
GPSLatitudeRef: b'N'
GPSLatitude: ((31, 1), (42, 1), (3916422, 100000))
GPSLongitudeRef: b'E'
GPSLongitude: ((34, 1), (47, 1), (3491355, 100000))
GPSAltitudeRef: 0
GPSAltitude: (15795, 100)
GPSTimeStamp: ((10, 1), (59, 1), (4000, 1000))
GPSMapDatum: b'WGS-84\x00'
GPSDateStamp: b'2021:03:07\x00'


In [22]:
import piexif
import os

# שנה לנתיב של תמונה אחת שעדכנת
IMAGE_PATH = r"/Users/snirtahasa/Thesis/2021-03-07/DSC07845.JPG" 

def deep_dump_gps():
    print(f"--- Deep EXIF Dump for {os.path.basename(IMAGE_PATH)} ---")
    try:
        exif_dict = piexif.load(IMAGE_PATH)
        gps_ifd = exif_dict.get("GPS", {})
        
        for tag_id, value in gps_ifd.items():
            tag_name = piexif.TAGS["GPS"].get(tag_id, {}).get("name", str(tag_id))
            print(f"{tag_name} (ID: {tag_id}): {value} | Type: {type(value)}")
            
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    deep_dump_gps()

--- Deep EXIF Dump for DSC07845.JPG ---
GPSVersionID (ID: 0): (2, 3, 0, 0) | Type: <class 'tuple'>
GPSLatitudeRef (ID: 1): b'N' | Type: <class 'bytes'>
GPSLatitude (ID: 2): ((31, 1), (42, 1), (39164227, 1000000)) | Type: <class 'tuple'>
GPSLongitudeRef (ID: 3): b'E' | Type: <class 'bytes'>
GPSLongitude (ID: 4): ((34, 1), (47, 1), (34913556, 1000000)) | Type: <class 'tuple'>
GPSAltitudeRef (ID: 5): 0 | Type: <class 'int'>
GPSAltitude (ID: 6): (15795, 100) | Type: <class 'tuple'>
GPSMapDatum (ID: 18): b'WGS-84' | Type: <class 'bytes'>


In [19]:
exiftool -gpslatitude=31.xxxx -gpslongitude=34.xxxx -gpslatituderef=N -gpslongituderef=E -gpsaltitude=xxx DSC07845.JPG

SyntaxError: invalid decimal literal (3337591887.py, line 1)

In [8]:
import os
import shutil
import piexif
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# =============================================================================
# הגדרות משתמש
# =============================================================================

LOG_FILES = [
    r"/Users/snirtahasa/Thesis/DJI LOG/March-7-2021-12-56-05-Flight-Airdata.csv",
    r"/Users/snirtahasa/Thesis/DJI LOG/March-7-2021-13-15-59-Flight-Airdata.csv"
]

# ✅ תיקון באג: התמונות ב-photos/ לא בשורש התיקייה
IMAGE_FOLDER  = r"/Users/snirtahasa/Thesis/2021-03-07/photos"
BACKUP_FOLDER = r"/Users/snirtahasa/Thesis/2021-03-07/photos_ORIGINAL_BACKUP"

# קיזוז שעון המצלמה (שניות). המצלמה רצה על UTC, הקיזוז הוא drift נטו.
# ערך שלילי = שעון המצלמה קדם ל-GPS בX שניות.
CAMERA_CLOCK_OFFSET = -19.0


# =============================================================================
# כלי עזר: המרת קואורדינטה לפורמט EXIF Rational
# =============================================================================

def float_to_rational(value: float) -> tuple:
    """
    ממיר ערך עשרוני (31.7079) לפורמט EXIF:
    ((degrees,1), (minutes,1), (seconds_num, 1_000_000))
    דיוק: 6 ספרות אחרי נקודה ≈ ~1 ס"מ
    """
    if pd.isna(value):
        return ((0, 1), (0, 1), (0, 1))
    value   = abs(float(value))
    deg     = int(value)
    min_f   = (value - deg) * 60
    minutes = int(min_f)
    sec_f   = (min_f - minutes) * 60
    return ((deg, 1), (minutes, 1), (int(round(sec_f * 1_000_000)), 1_000_000))


# =============================================================================
# כתיבת GPS EXIF — פורמט מלא שArcGIS קורא נכון
# =============================================================================

def inject_gps_to_jpg(image_path: str, lat: float, lon: float, alt: float) -> bool:
    """
    כותב GPS EXIF לקובץ JPEG בפורמט תקני מלא.

    שלושת הדברים שגרמו ל-0,0 באפריקה בגרסה הישנה:
      1. חסר GPSVersionID  — ArcGIS מתעלם מבלוק GPS ללא tag זה
      2. GPSAltitudeRef כ-int — משחית את מבנה ה-IFD כולו
      3. חסר GPSMapDatum  — ArcGIS לא יודע באיזו datum להשתמש
      4. piexif.insert בלי remove — מוסיף APP1 חדש לצד הישן,
         ArcGIS קורא את הראשון (בלי GPS)
    """
    try:
        exif_dict = piexif.load(image_path)
    except Exception as e:
        print(f"    ❌ לא הצלחתי לפתוח EXIF: {os.path.basename(image_path)} — {e}")
        return False

    lat_ref = b'N' if lat >= 0 else b'S'
    lon_ref = b'E' if lon >= 0 else b'W'

    gps_ifd = {
        piexif.GPSIFD.GPSVersionID:    (2, 3, 0, 0),
        piexif.GPSIFD.GPSLatitudeRef:  lat_ref,
        piexif.GPSIFD.GPSLatitude:     float_to_rational(abs(lat)),
        piexif.GPSIFD.GPSLongitudeRef: lon_ref,
        piexif.GPSIFD.GPSLongitude:    float_to_rational(abs(lon)),
        piexif.GPSIFD.GPSAltitudeRef:  b'\x00' if alt >= 0 else b'\x01',
        piexif.GPSIFD.GPSAltitude:     (int(abs(alt) * 100), 100),
        piexif.GPSIFD.GPSMapDatum:     b'WGS-84',
    }

    try:
        exif_dict['GPS'] = gps_ifd
        exif_bytes = piexif.dump(exif_dict)
        piexif.remove(image_path)      # מנקה APP1 ישן — קריטי!
        piexif.insert(exif_bytes, image_path)
        return True
    except Exception as e:
        print(f"    ❌ כשלון כתיבת EXIF: {os.path.basename(image_path)} — {e}")
        return False


# =============================================================================
# אימות: קריאה חזרה אחרי כתיבה
# =============================================================================

def verify_gps_exif(image_path: str) -> tuple | None:
    """מחזיר (lat, lon) אם GPS נכתב תקין, אחרת None."""
    try:
        exif  = piexif.load(image_path)
        gps   = exif.get('GPS', {})
        lat_d = gps.get(piexif.GPSIFD.GPSLatitude)
        lon_d = gps.get(piexif.GPSIFD.GPSLongitude)
        lat_r = gps.get(piexif.GPSIFD.GPSLatitudeRef)
        lon_r = gps.get(piexif.GPSIFD.GPSLongitudeRef)
        if not all([lat_d, lon_d, lat_r, lon_r]):
            return None
        def r2f(r):
            return r[0][0]/r[0][1] + r[1][0]/r[1][1]/60 + r[2][0]/r[2][1]/3600
        lat = r2f(lat_d) * (1 if lat_r == b'N' else -1)
        lon = r2f(lon_d) * (1 if lon_r == b'E' else -1)
        return (lat, lon)
    except Exception:
        return None


# =============================================================================
# לוגיקה ראשית: קריאת לוגים + אינטרפולציה + הזרקה
# =============================================================================

def run_smart_geosync():
    print("=" * 60)
    print("  DJI GeoSync  ·  Linear Interpolation  ·  ArcGIS-Ready")
    print("=" * 60)

    # ── גיבוי ──────────────────────────────────────────────────────
    if not os.path.exists(BACKUP_FOLDER):
        print(f"\n📦 מגבה תמונות מקוריות...")
        shutil.copytree(IMAGE_FOLDER, BACKUP_FOLDER)
        print(f"   שמור ב: {BACKUP_FOLDER}")
    else:
        print(f"\nℹ️  גיבוי קיים — ממשיך.")

    # ── טעינת לוגי GPS ─────────────────────────────────────────────
    print("\n1. קורא לוגי DJI...")
    df_list = []
    for f in LOG_FILES:
        if os.path.exists(f):
            df_list.append(pd.read_csv(f))
            print(f"   ✔ {os.path.basename(f)}")
        else:
            print(f"   ⚠️  לא נמצא: {f}")

    if not df_list:
        print("❌ אין לוגים. עוצר.")
        return

    df_logs = pd.concat(df_list, ignore_index=True)
    df_logs['exact_time'] = (
        pd.to_datetime(df_logs['datetime(utc)'])
        + pd.to_timedelta(df_logs['time(millisecond)'], unit='ms')
    )
    df_logs = (
        df_logs
        .sort_values('exact_time')
        .dropna(subset=['latitude', 'longitude'])
        .astype({'exact_time': 'datetime64[ns]'})
    )

    gps_start = df_logs['exact_time'].iloc[0]
    gps_end   = df_logs['exact_time'].iloc[-1]
    print(f"   טווח GPS: {gps_start} → {gps_end} UTC")
    print(f"   סה\"כ {len(df_logs):,} נקודות GPS (10 Hz)")

    # ── קריאת תמונות ───────────────────────────────────────────────
    print("\n2. קורא EXIF תמונות...")
    image_files = sorted([
        f for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith('.jpg')
    ])
    print(f"   נמצאו {len(image_files)} קבצי JPG")

    images_data = []
    errors      = []
    subsec_count = 0

    for img in image_files:
        path = os.path.join(IMAGE_FOLDER, img)
        try:
            exif_dict = piexif.load(path)
            exif_ifd  = exif_dict.get("Exif", {})

            dt_bytes = exif_ifd.get(piexif.ExifIFD.DateTimeOriginal)
            if not dt_bytes:
                errors.append(f"חסר DateTimeOriginal: {img}")
                continue

            dt_str = dt_bytes.decode('utf-8').strip('\x00')
            dt = datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S")

            # SubSecTimeOriginal: Sony Alpha כותבת תת-שנייה אמיתית (למשל b'5' = 0.5 שנייה)
            # זה מחליף את ניחוש ה-ms_offset ונותן דיוק של ~10ms
            subsec_bytes = exif_ifd.get(piexif.ExifIFD.SubSecTimeOriginal)
            if subsec_bytes:
                subsec_str = subsec_bytes.decode('utf-8').strip('\x00')
                # Sony כותבת "5" = 0.5 שנייה, או "50" = 0.50 שנייה
                subsec_float = float("0." + subsec_str) if subsec_str.isdigit() else 0.0
                dt = dt + timedelta(seconds=subsec_float)
                subsec_count += 1

            images_data.append({'image_name': img, 'base_time': dt, 'has_subsec': subsec_bytes is not None})
        except Exception as e:
            errors.append(f"שגיאת EXIF {img}: {e}")

    if subsec_count > 0:
        print(f"   ✅ {subsec_count}/{len(image_files)} תמונות עם SubSecTimeOriginal — דיוק תת-שנייתי")
    else:
        print(f"   ℹ️  אין SubSecTimeOriginal — משתמש בפיזור אחיד תוך-שנייתי")

    if errors:
        print(f"   ⚠️  {len(errors)} תמונות עם בעיות EXIF:")
        for e in errors[:5]:
            print(f"      - {e}")
        if len(errors) > 5:
            print(f"      ... ועוד {len(errors)-5}")

    if not images_data:
        print("❌ אף תמונה לא נקראה בהצלחה. עוצר.")
        return

    df_images = pd.DataFrame(images_data).sort_values('base_time')

    # ── פיזור תוך-שנייתי: SubSecTimeOriginal אם קיים, אחרת ניחוש ──
    # SubSecTimeOriginal כבר הוחל בלולאה למעלה (base_time כולל תת-שנייה)
    # הפיזור האחיד רלוונטי רק לתמונות ללא SubSec
    df_images['has_subsec'] = df_images.get('has_subsec', False)

    if not df_images['has_subsec'].any():
        # אף תמונה אין SubSec — פיזור אחיד
        df_images['count'] = df_images.groupby('base_time').cumcount()
        df_images['total'] = df_images.groupby('base_time')['image_name'].transform('count')
        df_images['base_time'] = df_images['base_time'] + pd.to_timedelta(
            df_images['count'] / df_images['total'], unit='s'
        )

    df_images['exact_time'] = (
        df_images['base_time']
        + pd.to_timedelta(CAMERA_CLOCK_OFFSET, unit='s')
    )
    df_images = df_images.astype({'exact_time': 'datetime64[ns]'})

    # ── בדיקת טווח: אזהרה על תמונות מחוץ לחלון ה-GPS ──────────────
    out_of_range = df_images[
        (df_images['exact_time'] < gps_start) |
        (df_images['exact_time'] > gps_end)
    ]
    if len(out_of_range) > 0:
        print(f"\n   ⚠️  {len(out_of_range)} תמונות מחוץ לטווח GPS (לא יקבלו מיקום):")
        for _, r in out_of_range.iterrows():
            print(f"      - {r['image_name']}  ({r['exact_time']})")

    # ── אינטרפולציה ליניארית לפי ציר זמן ──────────────────────────
    print("\n3. אינטרפולציה ליניארית על ציר GPS+תמונות...")

    logs_s  = df_logs[['exact_time','latitude','longitude',
                        'altitude_above_seaLevel(meters)']].copy()
    logs_s['type'] = 'GPS'

    imgs_s  = df_images[['exact_time','image_name']].copy()
    imgs_s['type'] = 'IMAGE'

    combined = (
        pd.concat([logs_s, imgs_s])
        .sort_values('exact_time')
        .set_index('exact_time')
    )

    # אינטרפולציה רק בין נקודות — לא חוץ מהטווח (limit_direction='forward' בלבד)
    for col in ['latitude', 'longitude', 'altitude_above_seaLevel(meters)']:
        combined[col] = combined[col].interpolate(
            method='time',
            limit_direction='forward',   # ✅ לא מחוץ לטווח GPS
            limit_area='inside'          # ✅ רק בין נקודות קיימות
        )

    final_images = (
        combined[combined['type'] == 'IMAGE']
        .dropna(subset=['latitude', 'longitude'])
        .reset_index()
    )

    skipped = len(df_images) - len(final_images)
    print(f"   {len(final_images)} תמונות יקבלו GPS, {skipped} ידולגו (מחוץ לטווח)")

    # ── הזרקת EXIF ─────────────────────────────────────────────────
    print(f"\n4. מזריק GPS EXIF ל-{len(final_images)} תמונות...")
    success, failed, verify_fail = 0, 0, 0

    for _, row in final_images.iterrows():
        path = os.path.join(IMAGE_FOLDER, row['image_name'])
        lat  = row['latitude']
        lon  = row['longitude']
        alt  = row['altitude_above_seaLevel(meters)']

        if inject_gps_to_jpg(path, lat, lon, alt):
            result = verify_gps_exif(path)
            if result is None:
                print(f"    ⚠️  נכתב אבל לא נקרא חזרה: {row['image_name']}")
                verify_fail += 1
            else:
                success += 1
        else:
            failed += 1

    # ── ייצוא CSV (fallback ל-XY Table to Point ב-ArcGIS) ─────────
    csv_path = os.path.join(IMAGE_FOLDER, "gps_coordinates.csv")
    export_cols = ['image_name', 'exact_time', 'latitude', 'longitude',
                   'altitude_above_seaLevel(meters)']
    final_images[export_cols].rename(columns={
        'altitude_above_seaLevel(meters)': 'altitude_m',
        'exact_time': 'datetime_utc'
    }).to_csv(csv_path, index=False)

    # ── סיכום ──────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print(f"  ✅ הצליח:            {success} תמונות")
    if failed:      print(f"  ❌ כשל כתיבה:       {failed} תמונות")
    if verify_fail: print(f"  ⚠️  כשל אימות:       {verify_fail} תמונות")
    if skipped:     print(f"  ⏭️  מחוץ לטווח GPS: {skipped} תמונות")
    print(f"\n  📊 CSV גם נשמר ב: {csv_path}")
    print(f"     (לשימוש עם XY Table to Point אם הגרירה לא עובדת)")
    print("=" * 60)
    print("\n✅ מוכן לייבוא ל-ArcGIS → GeoTagged Photos to Points")
    print("   Coordinate System: GCS_WGS_1984")


if __name__ == "__main__":
    run_smart_geosync()

  DJI GeoSync  ·  Linear Interpolation  ·  ArcGIS-Ready

ℹ️  גיבוי קיים — ממשיך.

1. קורא לוגי DJI...
   ✔ March-7-2021-12-56-05-Flight-Airdata.csv
   ✔ March-7-2021-13-15-59-Flight-Airdata.csv
   טווח GPS: 2021-03-07 10:56:05.100000 → 2021-03-07 11:34:24.600000 UTC
   סה"כ 15,986 נקודות GPS (10 Hz)

2. קורא EXIF תמונות...
   נמצאו 960 קבצי JPG
   ℹ️  אין SubSecTimeOriginal — משתמש בפיזור אחיד תוך-שנייתי

   ⚠️  3 תמונות מחוץ לטווח GPS (לא יקבלו מיקום):
      - DSC07752.JPG  (2021-03-07 10:56:00)
      - DSC07753.JPG  (2021-03-07 10:56:03)
      - DSC07754.JPG  (2021-03-07 10:56:05)

3. אינטרפולציה ליניארית על ציר GPS+תמונות...
   957 תמונות יקבלו GPS, 3 ידולגו (מחוץ לטווח)

4. מזריק GPS EXIF ל-957 תמונות...

  ✅ הצליח:            957 תמונות
  ⏭️  מחוץ לטווח GPS: 3 תמונות

  📊 CSV גם נשמר ב: /Users/snirtahasa/Thesis/2021-03-07/photos/gps_coordinates.csv
     (לשימוש עם XY Table to Point אם הגרירה לא עובדת)

✅ מוכן לייבוא ל-ArcGIS → GeoTagged Photos to Points
   Coordinate System: GCS_W

In [10]:
"""
GeoSync — DJI Flight Log → Sony EXIF GPS Injector
===================================================
Core innovations in this version:

1. AUTO-CALIBRATION of CAMERA_CLOCK_OFFSET
   Uses the drone's own isPhoto=1 events as GPS ground truth.
   Scans candidate offsets and finds the one that minimises
   the distance between each Sony photo's interpolated GPS
   and the nearest DJI trigger position.

2. SPEED-DISTANCE CORRELATION (fallback calibration)
   For each candidate offset: measures how well drone speed
   at photo-time correlates with distance to the next photo.
   Peaks sharply at the correct offset.

3. VELOCITY CORRECTION using xSpeed / ySpeed (world frame)
   Compensates for camera shutter latency using the actual
   East/North velocity vector — no heading-trig approximation.

4. POSITION UNCERTAINTY per photo in exported CSV.
"""

import os
import math
import shutil
import piexif
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ============================================================
# USER SETTINGS
# ============================================================

LOG_FILES = [
    r"/Users/snirtahasa/Thesis/DJI LOG/March-7-2021-12-56-05-Flight-Airdata.csv",
    r"/Users/snirtahasa/Thesis/DJI LOG/March-7-2021-13-15-59-Flight-Airdata.csv",
]
IMAGE_FOLDER  = r"/Users/snirtahasa/Thesis/2021-03-07/photos"
BACKUP_FOLDER = r"/Users/snirtahasa/Thesis/2021-03-07/photos_ORIGINAL_BACKUP"

# Starting guess (seconds). Auto-calibration will refine this.
CAMERA_CLOCK_OFFSET_SECONDS = -19.0

# Set only if camera is in local time (e.g. Israel UTC+2 → set 2).
# If camera is on UTC (common when synced to DJI GPS), leave 0.
CAMERA_TIMEZONE_OFFSET_HOURS = 0

# Camera mechanical shutter latency (ms). Drone keeps moving during this.
# Typical Sony mirrorless: 50–150 ms.
CAMERA_SHUTTER_LATENCY_MS = 100

# Auto-calibration search window around the starting guess (seconds).
CALIBRATION_SEARCH_WINDOW = 15.0   # scan ±15 s
CALIBRATION_STEP_COARSE   = 0.5    # coarse pass resolution
CALIBRATION_STEP_FINE     = 0.05   # fine pass resolution (around coarse best)

# Photos with positional uncertainty above this are flagged in the CSV.
# At 5 m/s cruise + 0.5s EXIF precision → 2.5m is the expected floor.
# Set to 5.0 so HIGH flags only appear for genuinely fast segments (>10 m/s).
UNCERTAINTY_FLAG_M = 5.0


# ============================================================
# UTILITIES
# ============================================================

def backup_images():
    if os.path.exists(BACKUP_FOLDER):
        print(f"  [backup] already exists — skipping.")
        return
    print(f"  [backup] copying to {BACKUP_FOLDER} ...")
    shutil.copytree(IMAGE_FOLDER, BACKUP_FOLDER)
    print(f"  [backup] done.")


def float_to_rational(value: float) -> tuple:
    """Decimal degrees → EXIF rational. 1e-6 denominator ≈ 1 cm precision."""
    if pd.isna(value):
        return ((0, 1), (0, 1), (0, 1))
    value   = abs(float(value))
    deg     = int(value)
    min_f   = (value - deg) * 60
    minutes = int(min_f)
    sec_f   = (min_f - minutes) * 60
    return ((deg, 1), (minutes, 1), (int(round(sec_f * 1_000_000)), 1_000_000))


def inject_gps_to_jpg(image_path: str, lat: float, lon: float, alt: float) -> bool:
    try:
        exif_dict = piexif.load(image_path)
    except Exception as e:
        print(f"    [ERROR] cannot read EXIF {os.path.basename(image_path)}: {e}")
        return False

    gps_ifd = {
        piexif.GPSIFD.GPSVersionID:    (2, 3, 0, 0),
        piexif.GPSIFD.GPSLatitudeRef:  b'N' if lat >= 0 else b'S',
        piexif.GPSIFD.GPSLatitude:     float_to_rational(abs(lat)),
        piexif.GPSIFD.GPSLongitudeRef: b'E' if lon >= 0 else b'W',
        piexif.GPSIFD.GPSLongitude:    float_to_rational(abs(lon)),
        piexif.GPSIFD.GPSAltitudeRef:  b'\x00' if alt >= 0 else b'\x01',
        piexif.GPSIFD.GPSAltitude:     (int(abs(alt) * 100), 100),
        piexif.GPSIFD.GPSMapDatum:     b'WGS-84',
    }
    try:
        exif_dict['GPS'] = gps_ifd
        exif_bytes = piexif.dump(exif_dict)
        piexif.remove(image_path)
        piexif.insert(exif_bytes, image_path)
        return True
    except Exception as e:
        print(f"    [ERROR] EXIF write failed {os.path.basename(image_path)}: {e}")
        return False


# ============================================================
# GPS INTERPOLATION HELPER
# ============================================================

def detect_flight_gaps(df_logs: pd.DataFrame,
                       gap_threshold_s: float = 30.0) -> list[tuple]:
    """
    Finds gaps between separate flights within the merged log.
    A gap is any interval between consecutive GPS points longer than
    gap_threshold_s seconds (typically happens between battery swaps).
    Returns a list of (gap_start, gap_end) as datetime64 pairs.
    """
    times  = df_logs['exact_time'].sort_values().values
    deltas = pd.Series(times[1:]) - pd.Series(times[:-1])
    gaps   = []
    for i, d in enumerate(deltas):
        if d > np.timedelta64(int(gap_threshold_s * 1e9), 'ns'):
            gaps.append((times[i], times[i + 1]))
    return gaps


def interpolate_gps(df_images_timed: pd.DataFrame,
                    df_logs: pd.DataFrame,
                    verbose: bool = True) -> pd.DataFrame:
    """
    Merge image timestamps into the GPS timeline and interpolate
    lat/lon/alt/xSpeed/ySpeed/speed for each image.

    Gap-aware: images that fall inside a between-flight gap
    (drone on ground between battery swaps) are excluded so they
    don't silently inherit the home-point coordinates.

    Window-safe: images outside [gps_start, gps_end] are removed
    BEFORE interpolation. This is more reliable than limit_area='inside'
    which silently extrapolates in some pandas versions when limit= is unset.
    """
    gps_start = df_logs['exact_time'].iloc[0]
    gps_end   = df_logs['exact_time'].iloc[-1]

    # Hard clip: drop images outside the GPS window before interpolating
    in_window = (
        (df_images_timed['exact_time'] >= gps_start) &
        (df_images_timed['exact_time'] <= gps_end)
    )
    n_outside = (~in_window).sum()
    if n_outside and verbose:
        print(f"  ✂  {n_outside} images clipped — outside GPS window "
              f"({gps_start.strftime('%H:%M:%S')}–{gps_end.strftime('%H:%M:%S')} UTC)")
    df_images_timed = df_images_timed[in_window].copy()

    # Detect gaps between separate flights (e.g. battery swap)
    gaps = detect_flight_gaps(df_logs, gap_threshold_s=30.0)

    cols_gps = ['exact_time', 'latitude', 'longitude',
                'altitude_above_seaLevel(meters)',
                'xSpeed(m/s)', 'ySpeed(m/s)', 'speed(m/s)', 'flycState']

    logs_s   = df_logs[cols_gps].copy();  logs_s['_type'] = 'GPS'
    images_s = df_images_timed[['exact_time', 'image_name']].copy()
    images_s['_type'] = 'IMAGE'

    combined = (
        pd.concat([logs_s, images_s])
          .sort_values('exact_time')
          .set_index('exact_time')
    )
    for col in ['latitude', 'longitude', 'altitude_above_seaLevel(meters)',
                'xSpeed(m/s)', 'ySpeed(m/s)', 'speed(m/s)']:
        # limit=len(combined) ensures all interior gaps are filled;
        # images outside the window were already removed above so no
        # extrapolation is possible here.
        combined[col] = combined[col].interpolate(method='time',
                                                   limit=len(combined))

    # Forward-fill flycState so each image row inherits the drone's flight
    # phase at its interpolated time (Waypoint / Go_Home / AutoLanding …)
    combined['flycState'] = combined['flycState'].ffill()

    # Null-out any image that falls inside a between-flight gap
    # so it is excluded by the subsequent dropna() call
    for gap_start, gap_end in gaps:
        mask = (
            (combined.index > gap_start) &
            (combined.index < gap_end) &
            (combined['_type'] == 'IMAGE')
        )
        combined.loc[mask, ['latitude', 'longitude']] = np.nan

    result = (
        combined[combined['_type'] == 'IMAGE']
          .dropna(subset=['latitude', 'longitude'])
          .reset_index()
    )
    return result, gaps


# ============================================================
# AUTO-CALIBRATION
# ============================================================

def _apply_offset(df_images_base: pd.DataFrame, offset_s: float) -> pd.DataFrame:
    df = df_images_base.copy()
    df['exact_time'] = (
        df['base_time']
        + pd.to_timedelta(df['subsec'], unit='s')
        + pd.to_timedelta(offset_s, unit='s')
    ).astype('datetime64[ns]')
    return df


def score_isphoto(offset_s: float,
                  df_images_base: pd.DataFrame,
                  df_logs: pd.DataFrame,
                  isphoto_rows: pd.DataFrame) -> float:
    """
    For each DJI isPhoto=1 event we know:
      - exact UTC time the drone trigger fired
      - exact GPS position at that moment

    For a given clock offset, find the Sony photo closest in time
    to each trigger event and measure how far its interpolated GPS
    is from the trigger GPS.

    Returns mean error in meters (lower = better offset).
    Only valid when len(isphoto_rows) >= 2.
    """
    if len(isphoto_rows) < 2:
        return np.inf

    df_img = _apply_offset(df_images_base, offset_s)
    result, _ = interpolate_gps(df_img, df_logs, verbose=False)
    if len(result) < 3:
        return np.inf

    # Build a lookup: image_name → interpolated GPS
    gps_lookup = result.set_index('image_name')[['latitude', 'longitude']].to_dict('index')

    # For each isPhoto event, find the nearest Sony photo (by time)
    img_times = df_img.set_index('exact_time')['image_name']
    errors = []
    for _, ev in isphoto_rows.iterrows():
        # Find closest Sony image in time to this DJI trigger
        t_dji = ev['exact_time']
        idx   = (img_times.index - t_dji).abs().argmin()
        img   = img_times.iloc[idx]
        if img not in gps_lookup:
            continue
        g     = gps_lookup[img]
        # Haversine-lite error (metres)
        dlat  = (g['latitude']  - ev['latitude'])  * 111_111
        dlon  = (g['longitude'] - ev['longitude']) * 111_111 * math.cos(math.radians(ev['latitude']))
        errors.append(math.sqrt(dlat**2 + dlon**2))

    return float(np.mean(errors)) if errors else np.inf


def score_speed_distance(offset_s: float,
                         df_images_base: pd.DataFrame,
                         df_logs: pd.DataFrame) -> float:
    """
    Fallback calibration: measures correlation between drone speed at
    photo-capture time and the GPS distance to the NEXT photo.

    When the offset is correct:
      fast drone  → large inter-photo gap
      slow drone  → small inter-photo gap
    → Pearson correlation is maximised.

    Returns negative correlation (so minimising == finding best offset).
    """
    df_img = _apply_offset(df_images_base, offset_s)
    result, _ = interpolate_gps(df_img, df_logs, verbose=False)
    if len(result) < 5:
        return 0.0

    # Use only mid-flight photos: drone airborne at mission altitude
    # and actually moving. Ground photos corrupt the correlation signal.
    ground_elev  = df_logs['altitude_above_seaLevel(meters)'].iloc[0]  # takeoff altitude
    min_alt_asl  = ground_elev + 50.0    # at least 50 m above ground
    min_speed    = 1.5                   # m/s — ignore hover/landing
    flight_mask  = (
        (result['altitude_above_seaLevel(meters)'] > min_alt_asl) &
        (result['speed(m/s)'] > min_speed)
    )
    result = result[flight_mask].reset_index(drop=True)

    if len(result) < 10:
        return 0.0

    lats   = result['latitude'].values
    lons   = result['longitude'].values
    speeds = result['speed(m/s)'].values

    d_lat  = np.diff(lats) * 111_111
    d_lon  = np.diff(lons) * 111_111 * np.cos(np.radians(np.mean(lats)))
    dists  = np.sqrt(d_lat**2 + d_lon**2)
    avg_sp = (speeds[:-1] + speeds[1:]) / 2

    if np.std(dists) < 1e-6 or np.std(avg_sp) < 1e-6:
        return 0.0
    corr = np.corrcoef(avg_sp, dists)[0, 1]
    return -float(corr) if not np.isnan(corr) else 0.0


def auto_calibrate(df_images_base: pd.DataFrame,
                   df_logs: pd.DataFrame,
                   isphoto_rows: pd.DataFrame,
                   manual_offset: float) -> float:
    """
    Two-pass offset search:
      Pass 1 (coarse): scan ±CALIBRATION_SEARCH_WINDOW in 0.5 s steps
      Pass 2 (fine):   scan ±2 s around coarse best in 0.05 s steps

    Uses isPhoto score when available (≥2 triggers), otherwise
    falls back to speed-distance correlation.
    """
    use_isphoto = len(isphoto_rows) >= 2
    method_name = "isPhoto anchor" if use_isphoto else "speed-distance correlation"
    print(f"  calibration method : {method_name}")

    # Python ternary + lambda requires explicit parentheses on each branch
    if use_isphoto:
        score_fn = lambda o: score_isphoto(o, df_images_base, df_logs, isphoto_rows)
    else:
        score_fn = lambda o: score_speed_distance(o, df_images_base, df_logs)

    # ── coarse pass ──────────────────────────────────────
    coarse_offsets = np.arange(
        manual_offset - CALIBRATION_SEARCH_WINDOW,
        manual_offset + CALIBRATION_SEARCH_WINDOW + CALIBRATION_STEP_COARSE,
        CALIBRATION_STEP_COARSE
    )
    coarse_scores = [score_fn(o) for o in coarse_offsets]
    best_coarse   = coarse_offsets[int(np.argmin(coarse_scores))]

    # ── fine pass ────────────────────────────────────────
    fine_offsets = np.arange(
        best_coarse - 2.0,
        best_coarse + 2.0 + CALIBRATION_STEP_FINE,
        CALIBRATION_STEP_FINE
    )
    fine_scores = [score_fn(o) for o in fine_offsets]
    best_fine   = fine_offsets[int(np.argmin(fine_scores))]

    delta = best_fine - manual_offset
    print(f"  manual offset      : {manual_offset:.1f} s")
    print(f"  auto-calibrated    : {best_fine:.2f} s  (Δ = {delta:+.2f} s)")
    if use_isphoto:
        print(f"  best anchor error  : {min(fine_scores):.1f} m  "
              f"(across {len(isphoto_rows)} DJI trigger events)")

    # Safety check: speed-distance correlation works poorly for distance-triggered
    # surveys (inter-photo distance is constant → correlation signal is weak →
    # calibration may find a random local minimum far from the true offset).
    # If the result deviates more than 8 s from the manual guess without isPhoto
    # anchors, the signal is unreliable — fall back to the manual offset.
    MAX_AUTO_DEVIATION_S = 8.0
    if not use_isphoto and abs(delta) > MAX_AUTO_DEVIATION_S:
        print(f"\n  ⚠  Auto-calibration deviated {delta:+.2f} s from manual guess.")
        print(f"     Speed-distance correlation may be unreliable for this flight")
        print(f"     pattern (distance-triggered waypoint survey).")
        print(f"     ➜  Falling back to manual offset: {manual_offset:.1f} s")
        return manual_offset

    return best_fine


# ============================================================
# MAIN
# ============================================================

def run_smart_geosync():
    print("=" * 60)
    print("  GeoSync  ·  Auto-Calibrated  ·  Velocity-Corrected")
    print("=" * 60)

    backup_images()

    # ── 1. Load GPS logs ──────────────────────────────────────────
    print("\n[1/5] Loading DJI flight logs...")
    df_list = []
    for f in LOG_FILES:
        if os.path.exists(f):
            df = pd.read_csv(f)
            df.columns = df.columns.str.strip()   # remove DJI leading spaces
            df_list.append(df)
            print(f"  ✔ {os.path.basename(f)}")
        else:
            print(f"  ✗ not found: {f}")

    if not df_list:
        print("[ERROR] No logs found. Exiting."); return

    df_logs = pd.concat(df_list, ignore_index=True)
    # datetime(utc) already contains the wall-clock UTC time (1-second precision).
    # time(millisecond) is elapsed ms since flight start — do NOT add it directly.
    # We only need the sub-second remainder for 10 Hz precision.
    df_logs['exact_time'] = (
        pd.to_datetime(df_logs['datetime(utc)'])
        + pd.to_timedelta(df_logs['time(millisecond)'] % 1000, unit='ms')
    )
    df_logs = (
        df_logs
          .sort_values('exact_time')
          .dropna(subset=['latitude', 'longitude'])
          .astype({'exact_time': 'datetime64[ns]'})
    )
    gps_start = df_logs['exact_time'].iloc[0]
    gps_end   = df_logs['exact_time'].iloc[-1]
    print(f"  GPS window  : {gps_start}  →  {gps_end} UTC")
    print(f"  GPS points  : {len(df_logs):,}  (10 Hz)")

    # Extract DJI trigger events (isPhoto=1) — our calibration anchors
    isphoto_rows = df_logs[df_logs['isPhoto'] == 1][
        ['exact_time', 'latitude', 'longitude', 'speed(m/s)']
    ].reset_index(drop=True)
    print(f"  isPhoto events: {len(isphoto_rows)} "
          f"(GPS ground-truth anchors for calibration)")
    for _, r in isphoto_rows.iterrows():
        print(f"    {r['exact_time']}  lat={r['latitude']:.6f}  "
              f"lon={r['longitude']:.6f}  speed={r['speed(m/s)']:.2f} m/s")

    # ── 2. Read image EXIF timestamps ─────────────────────────────
    print("\n[2/5] Reading image EXIF timestamps...")
    image_files = sorted([
        f for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith('.jpg')
    ])
    print(f"  Found {len(image_files)} JPG files")

    images_data, errors = [], []
    for img in image_files:
        path = os.path.join(IMAGE_FOLDER, img)
        try:
            exif_dict = piexif.load(path)
            exif_ifd  = exif_dict.get("Exif", {})
            dt_bytes  = exif_ifd.get(piexif.ExifIFD.DateTimeOriginal)
            if not dt_bytes:
                errors.append(f"missing DateTimeOriginal: {img}"); continue
            dt = datetime.strptime(dt_bytes.decode().strip('\x00'), "%Y:%m:%d %H:%M:%S")

            # SubSecTimeOriginal: Sony Alpha writes real sub-second precision
            subsec = 0.0
            ss     = exif_ifd.get(piexif.ExifIFD.SubSecTimeOriginal)
            if ss:
                s = ss.decode().strip('\x00')
                if s.isdigit():
                    subsec = float("0." + s)

            images_data.append({'image_name': img, 'base_time': dt, 'subsec': subsec})
        except Exception as e:
            errors.append(f"EXIF error {img}: {e}")

    if errors:
        print(f"  ⚠  {len(errors)} images with issues (first 3):")
        for e in errors[:3]: print(f"     {e}")
    if not images_data:
        print("[ERROR] No valid images found. Exiting."); return

    df_images = pd.DataFrame(images_data).sort_values('base_time').reset_index(drop=True)

    # Sub-second timing
    has_subsec = (df_images['subsec'] > 0).any()
    if has_subsec:
        print(f"  ✅ SubSecTimeOriginal present — sub-second timestamps used")
    else:
        print(f"  ℹ  No SubSecTimeOriginal — distributing evenly within each second")
        df_images['count'] = df_images.groupby('base_time').cumcount()
        df_images['total'] = df_images.groupby('base_time')['image_name'].transform('count')
        df_images['subsec'] = df_images['count'] / df_images['total']

    # Apply timezone offset to base_time (not the clock drift)
    tz_delta = CAMERA_TIMEZONE_OFFSET_HOURS * 3600
    df_images['base_time'] = df_images['base_time'] + pd.to_timedelta(tz_delta, unit='s')

    # ── 3. Auto-calibrate clock offset ───────────────────────────
    print("\n[3/5] Auto-calibrating CAMERA_CLOCK_OFFSET...")
    best_offset = auto_calibrate(
        df_images, df_logs, isphoto_rows, CAMERA_CLOCK_OFFSET_SECONDS
    )

    # Apply the calibrated offset
    df_images['exact_time'] = (
        df_images['base_time']
        + pd.to_timedelta(df_images['subsec'], unit='s')
        + pd.to_timedelta(best_offset, unit='s')
    ).astype('datetime64[ns]')

    # Warn about out-of-range images
    oor = df_images[
        (df_images['exact_time'] < gps_start) | (df_images['exact_time'] > gps_end)
    ]
    if len(oor):
        print(f"\n  ⚠  {len(oor)} images outside GPS window — will be skipped")

    # ── 4. Interpolate + velocity correction ─────────────────────
    print("\n[4/5] Interpolating GPS + applying velocity correction...")
    final_images, gaps = interpolate_gps(df_images, df_logs)

    if gaps:
        print(f"  Detected {len(gaps)} between-flight gap(s):")
        for gs, ge in gaps:
            dur = (pd.Timestamp(ge) - pd.Timestamp(gs)).total_seconds()
            print(f"    {pd.Timestamp(gs).strftime('%H:%M:%S')} → "
                  f"{pd.Timestamp(ge).strftime('%H:%M:%S')} UTC  "
                  f"({dur:.0f} s on ground — images in gap excluded)")

    # ── Mission-phase filter: keep only Waypoint images ──────────
    # Go_Home / AutoLanding / Motors_Started images are correctly GPS'd
    # but are not survey photos (drone returning home, ascending, racing
    # to first waypoint). Remove them so they don't appear in the mosaic.
    if 'flycState' in final_images.columns:
        mission_mask = final_images['flycState'].str.strip() == 'Waypoint'
        non_mission  = (~mission_mask).sum()
        if non_mission:
            states = final_images.loc[~mission_mask, 'flycState'].value_counts().to_dict()
            print(f"  ✈  {non_mission} non-survey images removed "
                  f"(flycState: {states})")
        final_images = final_images[mission_mask].reset_index(drop=True)

    # ── Altitude cross-validation ─────────────────────────────────
    # A photo whose interpolated GPS time maps to when the drone was
    # on the ground (<50 m AGL) must not receive flight coordinates.
    # This is the root cause of "some images look right then it loses it":
    # ground photos fall inside the GPS window and get random mid-flight GPS.
    ground_elev     = df_logs['altitude_above_seaLevel(meters)'].iloc[0]
    mission_alt_min = ground_elev + 50.0
    airborne_mask   = final_images['altitude_above_seaLevel(meters)'] > mission_alt_min
    ground_excluded = (~airborne_mask).sum()
    if ground_excluded:
        print(f"  ⚠  {ground_excluded} photos excluded — drone was on ground "
              f"at interpolated time (alt < {mission_alt_min:.0f} m ASL)")
    final_images = final_images[airborne_mask].reset_index(drop=True)

    skipped = len(df_images) - len(final_images)
    print(f"  {len(final_images)} in-flight images,  {skipped} skipped total")

    # Velocity correction: compensate for camera shutter latency
    # Position at shutter-open = GPS position + velocity × latency
    # Uses world-frame xSpeed (East) and ySpeed (North) directly.
    dt_s = CAMERA_SHUTTER_LATENCY_MS / 1000.0

    def apply_correction(row):
        v_east  = row['xSpeed(m/s)']
        v_north = row['ySpeed(m/s)']
        lat     = row['latitude']
        dlat = (v_north * dt_s) / 111_111
        dlon = (v_east  * dt_s) / (111_111 * math.cos(math.radians(lat)))
        spd  = row['speed(m/s)']
        # Uncertainty = ½s residual clock drift × speed (conservative estimate)
        uncertainty = round(spd * 0.5, 2)
        return pd.Series({
            'lat_final':  lat + dlat,
            'lon_final':  row['longitude'] + dlon,
            'speed_mps':  round(spd, 2),
            'uncertainty_m': uncertainty,
            'flag': 'HIGH' if uncertainty > UNCERTAINTY_FLAG_M else 'OK',
        })

    corrections = final_images.apply(apply_correction, axis=1)
    final_images = pd.concat([final_images, corrections], axis=1)

    n_high = (final_images['flag'] == 'HIGH').sum()
    if n_high:
        print(f"  ⚠  {n_high} photos flagged HIGH uncertainty "
              f"(speed > {UNCERTAINTY_FLAG_M / 0.5:.0f} m/s at capture)")

    # ── 5. Inject EXIF ───────────────────────────────────────────
    print(f"\n[5/5] Injecting GPS EXIF into {len(final_images)} images...")
    success, failed = 0, 0
    for _, row in final_images.iterrows():
        path = os.path.join(IMAGE_FOLDER, row['image_name'])
        ok   = inject_gps_to_jpg(path,
                                  row['lat_final'],
                                  row['lon_final'],
                                  row['altitude_above_seaLevel(meters)'])
        if ok: success += 1
        else:  failed  += 1

    # ── Export CSV ────────────────────────────────────────────────
    csv_path = os.path.join(IMAGE_FOLDER, "gps_coordinates.csv")
    final_images[[
        'image_name', 'exact_time',
        'lat_final', 'lon_final',
        'altitude_above_seaLevel(meters)',
        'speed_mps', 'uncertainty_m', 'flag'
    ]].rename(columns={
        'lat_final': 'latitude',
        'lon_final': 'longitude',
        'altitude_above_seaLevel(meters)': 'altitude_m',
        'exact_time': 'datetime_utc',
    }).to_csv(csv_path, index=False)

    # ── Summary ───────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print(f"  ✅ Success          : {success} images")
    if failed:          print(f"  ❌ Failed           : {failed} images")
    if ground_excluded: print(f"  🚫 Ground excluded  : {ground_excluded} (drone on ground at image time)")
    if skipped:         print(f"  ⏭  Skipped          : {skipped} (outside GPS window or on ground)")
    if n_high:          print(f"  ⚠  High uncertainty : {n_high} (fast flight, see CSV flag)")
    print(f"\n  Auto-calibrated offset: {best_offset:.2f} s")
    print(f"  Velocity correction  : {CAMERA_SHUTTER_LATENCY_MS} ms latency")
    print(f"  CSV: {csv_path}")
    print("=" * 60)
    print("\n  ArcGIS: GeoTagged Photos to Points | GCS_WGS_1984")
    print("  Tip: colour-code points by 'flag' column to see uncertainty")


if __name__ == "__main__":
    run_smart_geosync()

  GeoSync  ·  Auto-Calibrated  ·  Velocity-Corrected
  [backup] already exists — skipping.

[1/5] Loading DJI flight logs...
  ✔ March-7-2021-12-56-05-Flight-Airdata.csv
  ✔ March-7-2021-13-15-59-Flight-Airdata.csv
  GPS window  : 2021-03-07 10:56:05.100000  →  2021-03-07 11:25:12.600000 UTC
  GPS points  : 15,986  (10 Hz)
  isPhoto events: 0 (GPS ground-truth anchors for calibration)

[2/5] Reading image EXIF timestamps...
  Found 960 JPG files
  ℹ  No SubSecTimeOriginal — distributing evenly within each second

[3/5] Auto-calibrating CAMERA_CLOCK_OFFSET...
  calibration method : speed-distance correlation
  manual offset      : -19.0 s
  auto-calibrated    : -35.45 s  (Δ = -16.45 s)

  ⚠  Auto-calibration deviated -16.45 s from manual guess.
     Speed-distance correlation may be unreliable for this flight
     pattern (distance-triggered waypoint survey).
     ➜  Falling back to manual offset: -19.0 s

  ⚠  68 images outside GPS window — will be skipped

[4/5] Interpolating GPS + ap

In [2]:
import sys
sys.path.insert(0, '/Users/snirtahasa/Thesis')
from geosync_final import calc_offset_from_worldfile

calc_offset_from_worldfile(
    "/Users/snirtahasa/Library/Application Support/Claude/local-agent-mode-sessions/e408ffb8-c911-47df-aded-18f2a60fb283/0d13fc78-2d0e-476e-8498-2cd4cf5ef7df/local_5664cf64-bf56-4302-9cf9-305bb48c0faf/uploads/d76b508e-f73c-4d0f-9842-7af3171ffe84-1780854078688_8600.tfw",
    "/Users/snirtahasa/Thesis/DJI LOG/March-7-2021-13-15-59-Flight-Airdata.csv",
    "DSC08600.JPG",
)

  World-file image centre: lat=31.708257, lon=34.793689
  Nearest drone GPS time : 2021-03-07 11:21:45.200000  (dist=1.6 m)

  Camera EXIF time       : 2021-03-07 11:23:59
  Computed offset        : -133.8 s

  ➜  Add to CAMERA_CLOCK_OFFSET_PER_FILE:
     "March-7-2021-13-15-59-Flight-Airdata.csv": -133.8,


In [1]:
"""
inject_from_metashape.py
========================
Reads Metashape's camera-position CSV (lat, lon, alt, filename)
and injects those coordinates directly into the EXIF GPS tags
of each photo.

No offset calibration needed — Metashape already computed the
correct position for every image from photogrammetric alignment.
"""

import os
import piexif

# ── SETTINGS ──────────────────────────────────────────────────
METASHAPE_CSV  = r"/Users/snirtahasa/Thesis/2021-03-07//RGB_coordinates_information.csv"
IMAGE_FOLDER   = r"/Users/snirtahasa/Thesis/2021-03-07/photos"
OUTPUT_CSV     = r"/Users/snirtahasa/Thesis/2021-03-07/photos/gps_from_metashape.csv"
# ──────────────────────────────────────────────────────────────


def float_to_rational(value: float) -> tuple:
    value   = abs(float(value))
    deg     = int(value)
    min_f   = (value - deg) * 60
    minutes = int(min_f)
    sec_f   = (min_f - minutes) * 60
    return ((deg, 1), (minutes, 1), (int(round(sec_f * 1_000_000)), 1_000_000))


def inject_gps(image_path: str, lat: float, lon: float, alt: float) -> bool:
    try:
        exif_dict = piexif.load(image_path)
    except Exception as e:
        print(f"  [ERROR] cannot read EXIF {os.path.basename(image_path)}: {e}")
        return False

    gps_ifd = {
        piexif.GPSIFD.GPSVersionID:    (2, 3, 0, 0),
        piexif.GPSIFD.GPSLatitudeRef:  b'N' if lat >= 0 else b'S',
        piexif.GPSIFD.GPSLatitude:     float_to_rational(abs(lat)),
        piexif.GPSIFD.GPSLongitudeRef: b'E' if lon >= 0 else b'W',
        piexif.GPSIFD.GPSLongitude:    float_to_rational(abs(lon)),
        piexif.GPSIFD.GPSAltitudeRef:  b'\x00',
        piexif.GPSIFD.GPSAltitude:     (int(abs(alt) * 100), 100),
        piexif.GPSIFD.GPSMapDatum:     b'WGS-84',
    }
    try:
        exif_dict['GPS'] = gps_ifd
        exif_bytes = piexif.dump(exif_dict)
        piexif.remove(image_path)
        piexif.insert(exif_bytes, image_path)
        return True
    except Exception as e:
        print(f"  [ERROR] EXIF write failed {os.path.basename(image_path)}: {e}")
        return False


def run():
    print("=" * 55)
    print("  Metashape → EXIF GPS Injector")
    print("=" * 55)

    # Read CSV (no header: lat, lon, alt, filename)
    rows = []
    with open(METASHAPE_CSV, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) < 4:
                continue
            try:
                lat, lon, alt = float(parts[0]), float(parts[1]), float(parts[2])
                img = parts[3].strip()
                rows.append((lat, lon, alt, img))
            except ValueError:
                continue

    print(f"  Metashape entries : {len(rows)}")

    ok, skipped, missing = 0, 0, 0
    csv_lines = ["image_name,latitude,longitude,altitude_m"]

    for lat, lon, alt, img in rows:
        img_path = os.path.join(IMAGE_FOLDER, img)
        if not os.path.exists(img_path):
            missing += 1
            continue

        if inject_gps(img_path, lat, lon, alt):
            ok += 1
            csv_lines.append(f"{img},{lat:.10f},{lon:.10f},{alt:.4f}")
        else:
            skipped += 1

    # Write summary CSV
    with open(OUTPUT_CSV, 'w', encoding='utf-8') as f:
        f.write('\n'.join(csv_lines))

    print(f"\n  Injected  : {ok}")
    print(f"  Skipped   : {skipped}")
    print(f"  Not found : {missing}")
    print(f"\n  CSV saved : {OUTPUT_CSV}")
    print("\n  Done. Load photos in ArcGIS: GeoTagged Photos to Points.")


if __name__ == "__main__":
    run()

  Metashape → EXIF GPS Injector
  Metashape entries : 941

  Injected  : 941
  Skipped   : 0
  Not found : 0

  CSV saved : /Users/snirtahasa/Thesis/2021-03-07/photos/gps_from_metashape.csv

  Done. Load photos in ArcGIS: GeoTagged Photos to Points.
